<a href="https://colab.research.google.com/github/sumaiiiyyah/code/blob/master/PAPSS2_QSAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import requests
import pandas as pd

# Search terms for the target protein
search_terms = [
    "PAPSS2",
    "PAPS synthase 2",
    "3'-phosphoadenosine 5'-phosphosulfate synthase 2"
]

base_url = "https://www.ebi.ac.uk/chembl/api/data/target/search.json"

all_targets = []

for term in search_terms:
    response = requests.get(
        base_url,
        params={"q": term, "limit": 100},
        timeout=30
    )

    response.raise_for_status()

    results = response.json()["targets"]

    for target in results:
        all_targets.append(target)

# Remove duplicate target records
targets_df = pd.DataFrame(all_targets)

if not targets_df.empty:
    targets_df = targets_df.drop_duplicates(
        subset="target_chembl_id"
    )

    # Display important information
    columns = [
        "target_chembl_id",
        "pref_name",
        "organism",
        "target_type",
        "tax_id"
    ]

    print("Total unique target records:", len(targets_df))
    display(targets_df[columns])
else:
    print("No target records retrieved.")


Total unique target records: 165


,target_chembl_id,pref_name,organism,target_type,tax_id
0,CHEMBL4105790,Bifunctional 3'-phosphoadenosine 5'-phosphosul...,Homo sapiens,SINGLE PROTEIN,9606
2,CHEMBL2095188,Glycogen synthase kinase-3,Homo sapiens,PROTEIN FAMILY,9606
3,CHEMBL4523615,Glycogen synthase kinase-3 alpha/beta,Mus musculus,PROTEIN FAMILY,10090
4,CHEMBL262,Glycogen synthase kinase-3 beta,Homo sapiens,SINGLE PROTEIN,9606
5,CHEMBL2850,Glycogen synthase kinase-3 alpha,Homo sapiens,SINGLE PROTEIN,9606
...,...,...,...,...,...
193,CHEMBL5630,1-deoxy-D-xylulose 5-phosphate reductoisomerase,Mycobacterium tuberculosis,SINGLE PROTEIN,1773
195,CHEMBL2010623,1-deoxy-D-xylulose 5-phosphate reductoisomerase,Mycobacterium smegmatis (strain ATCC 700084 / ...,SINGLE PROTEIN,246196
196,CHEMBL2096664,Steroid 5-alpha-reductase,Rattus norvegicus,PROTEIN FAMILY,10116
197,CHEMBL2150844,Methionine synthase,Homo sapiens,SINGLE PROTEIN,9606


In [ ]:

import requests
import pandas as pd
import time

target_id = "CHEMBL4105790"

url = "https://www.ebi.ac.uk/chembl/api/data/activity.json"

all_activities = []
offset = 0
limit = 1000

while True:
    response = requests.get(
        url,
        params={
            "target_chembl_id": target_id,
            "limit": limit,
            "offset": offset
        },
        timeout=60
    )

    response.raise_for_status()
    data = response.json()

    batch = data.get("activities", [])
    all_activities.extend(batch)

    if len(batch) < limit:
        break

    offset += limit
    time.sleep(0.2)

activities_df = pd.DataFrame(all_activities)

print("Total activity records:", len(activities_df))

if not activities_df.empty:

    columns = [
        "molecule_chembl_id",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "pchembl_value",
        "assay_chembl_id",
        "assay_type",
        "document_chembl_id"
    ]

    available_columns = [
        col for col in columns
        if col in activities_df.columns
    ]

    display(activities_df[available_columns].head(30))

    print(
        "Unique compounds:",
        activities_df["molecule_chembl_id"].nunique()
    )

    print("\nActivity types:")
    print(activities_df["standard_type"].value_counts())

    print("\nActivity units:")
    print(activities_df["standard_units"].value_counts())

else:
    print("No activity records found for this target.")


Total activity records: 4


,molecule_chembl_id,standard_type,standard_relation,standard_value,standard_units,pchembl_value,assay_chembl_id,assay_type,document_chembl_id
0,CHEMBL4062193,EC50,>,10000.0,nM,None,CHEMBL4036829,B,CHEMBL4033760
1,CHEMBL4101680,EC50,>,10000.0,nM,None,CHEMBL4036829,B,CHEMBL4033760
2,CHEMBL4093377,EC50,>,10000.0,nM,None,CHEMBL4036829,B,CHEMBL4033760
3,CHEMBL4129274,Activity,None,None,None,None,CHEMBL4118716,B,CHEMBL4118021


Unique compounds: 4

Activity types:
standard_type
EC50        3
Activity    1
Name: count, dtype: int64

Activity units:
standard_units
nM    3
Name: count, dtype: int64


In [ ]:

# Retrieve assay information from ChEMBL

assay_ids = activities_df["assay_chembl_id"].dropna().unique()

assay_details = []

for assay_id in assay_ids:
    url = f"https://www.ebi.ac.uk/chembl/api/data/assay/{assay_id}.json"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    assay = response.json()

    assay_details.append({
        "assay_id": assay_id,
        "description": assay.get("description"),
        "assay_type": assay.get("assay_type"),
        "assay_category": assay.get("assay_category"),
        "document_id": assay.get("document_chembl_id"),
        "target_id": assay.get("target_chembl_id")
    })

assay_df = pd.DataFrame(assay_details)

display(assay_df)

# Save the retrieved records
activities_df.to_csv(
    "PAPSS2_ChEMBL_activity_raw.csv",
    index=False
)

assay_df.to_csv(
    "PAPSS2_ChEMBL_assay_details.csv",
    index=False
)

print("ChEMBL records saved successfully.")

,assay_id,description,assay_type,assay_category,document_id,target_id
0,CHEMBL4036829,Binding affinity to SK2 (unknown origin),B,None,CHEMBL4033760,CHEMBL4105790
1,CHEMBL4118716,Binding affinity to PAPSS2 in human NCI-H23 ce...,B,None,CHEMBL4118021,CHEMBL4105790


ChEMBL records saved successfully.


In [ ]:

document_ids = activities_df["document_chembl_id"].dropna().unique()

document_details = []

for doc_id in document_ids:
    url = f"https://www.ebi.ac.uk/chembl/api/data/document/{doc_id}.json"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    doc = response.json()

    document_details.append({
        "document_id": doc_id,
        "title": doc.get("title"),
        "authors": doc.get("authors"),
        "journal": doc.get("journal"),
        "year": doc.get("year"),
        "doi": doc.get("doi"),
        "pubmed_id": doc.get("pubmed_id"),
        "abstract": doc.get("abstract")
    })

documents_df = pd.DataFrame(document_details)

display(documents_df.T)


,0,1
document_id,CHEMBL4033760,CHEMBL4118021
title,Discovery of a Series of Indazole TRPA1 Antago...,Studies of TAK1-centered polypharmacology with...
authors,"Pryde DC, Marron BE, West CW, Reister S, Amato...","Tan L, Gurbani D, Weisberg EL, Jones DS, Rao S..."
journal,ACS Med Chem Lett,Bioorg Med Chem
year,2017,2017
doi,10.1021/acsmedchemlett.7b00140,10.1016/j.bmc.2016.11.034
pubmed_id,28626530,28038940
abstract,A series of TRPA1 antagonists is described whi...,Targeted polypharmacology provides an efficien...


In [ ]:

import requests

url = (
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
    "assay/target/genesymbol/PAPSS2/aids/JSON"
)

response = requests.get(url, timeout=60)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    data = response.json()

    assay_ids = data.get("IdentifierList", {}).get("AID", [])

    print("Total PubChem assay IDs:", len(assay_ids))
    print("Assay IDs:", assay_ids)

else:
    print("No assay IDs retrieved.")
    print(response.text[:500])


HTTP status: 200
Total PubChem assay IDs: 49
Assay IDs: [1622, 1904, 624099, 651810, 651811, 720703, 743121, 743454, 1053208, 1117269, 1117281, 1117357, 1159506, 1159578, 1159584, 1224826, 1224828, 1224830, 1259259, 1259260, 1259261, 1259264, 1259266, 1259267, 1259272, 1259274, 1259275, 1259276, 1259279, 1259280, 1259281, 1259282, 1259283, 1259284, 1259285, 1259286, 1259289, 1259290, 1259291, 1259297, 1259298, 1259299, 1259300, 1259306, 1259307, 1259323, 1454491, 1508587, 1508588]


In [ ]:

import requests
import pandas as pd
import time
import json

base = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid"

assay_audit = []

for aid in assay_ids:

    record = {
        "AID": aid,
        "summary": None,
        "targets": None,
        "error": None
    }

    try:
        # Retrieve assay summary
        r1 = requests.get(
            f"{base}/{aid}/summary/JSON",
            timeout=30
        )

        r1.raise_for_status()
        record["summary"] = r1.json()

        # Retrieve target annotations
        r2 = requests.get(
            f"{base}/{aid}/targets/GeneID,GeneSymbol,ProteinName/JSON",
            timeout=30
        )

        if r2.status_code == 200:
            record["targets"] = r2.json()

    except Exception as e:
        record["error"] = str(e)

    assay_audit.append(record)
    time.sleep(0.25)

# Preserve complete API responses
with open("PAPSS2_PubChem_assay_audit.json", "w") as f:
    json.dump(assay_audit, f, indent=2)

print("Assays inspected:", len(assay_audit))
print("Errors:", sum(x["error"] is not None for x in assay_audit))

# Inspect the first assay's response structure
print("\nFIRST ASSAY SUMMARY:")
print(json.dumps(assay_audit[0]["summary"], indent=2)[:5000])

print("\nFIRST ASSAY TARGET:")
print(json.dumps(assay_audit[0]["targets"], indent=2)[:3000])


Assays inspected: 49
Errors: 0

FIRST ASSAY SUMMARY:
{
  "AssaySummaries": {
    "AssaySummary": [
      {
        "AID": 1622,
        "SourceName": "Thermo Scientific Dharmacon RNAi Technologies",
        "SourceID": "luo_rgi_pilot",
        "Name": "RNAi Global Initiative pilot viability screen of human kinase and cell cycle genes",
        "Description": [
          "An RNAi-based silencing screen using SMARTpool siRNA silencing reagents (Thermo Fisher Scientific, Lafayette, CO, USA) targeting each of 859 human kinase and cell cycle genes that, when knocked down, induced apoptosis in HeLa cells (ATCC, CCL-2) was performed by the Luo lab at the Eppley Institute for Cancer Research, an RNAi Global Initiative member.  The assay for viability was performed using the CellTiter-Blue Cell Viability Assay (Promega, Madison, WI, USA), according to the manufacturer's protocol with slight modifications (See Technical Bulletin #TB317; www.promega.com/tbs/tb317/tb317.html). Genes whose knock-do

In [ ]:

import pandas as pd

rows = []

for item in assay_audit:

    aid = item["AID"]

    # Extract assay summary
    summaries = (
        item.get("summary", {})
        .get("AssaySummaries", {})
        .get("AssaySummary", [])
    )

    if isinstance(summaries, dict):
        summaries = [summaries]

    # Extract gene annotations
    information = (
        item.get("targets", {})
        .get("InformationList", {})
        .get("Information", [])
    )

    if isinstance(information, dict):
        information = [information]

    gene_ids = set()

    for info in information:
        gene_ids.update(info.get("GeneID", []))

    for summary in summaries:
        rows.append({
            "AID": aid,
            "Name": summary.get("Name"),
            "Source": summary.get("SourceName"),
            "Description": " ".join(
                summary.get("Description", [])
            )[:300],
            "PAPSS2_GeneID_9060": 9060 in gene_ids,
            "Gene_ID_count": len(gene_ids)
        })

pubchem_summary = pd.DataFrame(rows)

print("Total summary records:", len(pubchem_summary))

display(pubchem_summary[
    ["AID", "Name", "PAPSS2_GeneID_9060"]
])


Total summary records: 49


,AID,Name,PAPSS2_GeneID_9060
0,1622,RNAi Global Initiative pilot viability screen ...,True
1,1904,siRNA Circadian Assay,True
2,624099,An siRNA screen for human genes that are invol...,True
3,651810,Genome-wide high-content siRNA screens for reg...,True
4,651811,Genome-wide high-content siRNA screens for reg...,True
5,720703,AT1R-EGFR transactivation kinome screen - primary,True
6,743121,Druggable Genome siRNA Screen for Enhancers of...,True
7,743454,RNAi screen for vorinostat resistance genes - ...,True
8,1053208,A screen to identify the genes that modify the...,False
9,1117269,Human kinome siRNA screen in A549-BTR and MDA-...,True


In [ ]:

selected_aids = [1454491, 1259259, 1259323]

for aid in selected_aids:

    item = next(x for x in assay_audit if x["AID"] == aid)

    print("\n" + "="*70)
    print("ASSAY ID:", aid)

    summaries = (
        item["summary"]
        ["AssaySummaries"]["AssaySummary"]
    )

    for s in summaries:
        print("\nNAME:", s.get("Name"))
        print("\nDESCRIPTION:")
        print(" ".join(s.get("Description", []))[:3000])

        print("\nPROTOCOL:")
        print(" ".join(s.get("Protocol", []))[:1500])

    print("\nTARGET ANNOTATION:")
    print(item["targets"])



ASSAY ID: 1454491

NAME: Binding affinity to SK2 (unknown origin)

DESCRIPTION:
Title: Discovery of a Series of Indazole TRPA1 Antagonists.  Abstract: A series of TRPA1 antagonists is described which has as its core structure an indazole moiety. The physical properties and in vitro DMPK profiles are discussed. Good in vivo exposure was obtained with several analogs, allowing efficacy to be assessed in rodent models of inflammatory pain. Two compounds showed significant activity in these models when administered either systemically or topically. Protein chimeras were constructed to indicate compounds from the series bound in the S5 region of the channel, and a computational docking model was used to propose a binding mode for example compounds.

PROTOCOL:


TARGET ANNOTATION:
{'InformationList': {'Information': [{'AID': 1454491, 'GeneID': [9060], 'ProteinName': ["Bifunctional 3'-phosphoadenosine 5'-phosphosulfate synthase 2"], 'GeneSymbol': ['PAPSS2']}]}}

ASSAY ID: 1259259

NAME: A sc

In [ ]:

import requests
import json

url = "https://bindingdb.org/rest/getLigandsByUniprot"

params = {
    "uniprot": "O95340;100000",
    "response": "application/json"
}

response = requests.get(url, params=params, timeout=90)

print("HTTP status:", response.status_code)
print("Response size:", len(response.text), "characters")

if response.status_code == 200:

    bindingdb_raw = response.text

    with open("PAPSS2_BindingDB_raw.json", "w") as f:
        f.write(bindingdb_raw)

    try:
        bindingdb_data = response.json()

        print("\nResponse successfully parsed.")
        print("\nResponse preview:")
        print(json.dumps(bindingdb_data, indent=2)[:6000])

    except ValueError:
        print("\nResponse is not valid JSON.")
        print(response.text[:3000])

else:
    print(response.text[:1000])


HTTP status: 200
Response size: 266 characters

Response successfully parsed.

Response preview:
{
  "getLindsByUniprotResponse": {
    "bdb.hit": "0",
    "bdb.length": "NA",
    "bdb.uniprot_length": "614",
    "bdb.primary": "O95340",
    "bdb.alternative": [
      "O95340",
      "Q9BZL2",
      "Q9P0G6",
      "Q9UHM1",
      "Q9UKD3",
      "Q9UP30"
    ],
    "bdb.affinities": []
  }
}


In [ ]:

import pandas as pd

pubchem_classification = []

for item in assay_audit:

    aid = item["AID"]

    summaries = item["summary"]["AssaySummaries"]["AssaySummary"]

    for assay in summaries:

        name = assay.get("Name", "")
        description = " ".join(assay.get("Description", []))
        protocol = " ".join(assay.get("Protocol", []))

        text = (name + " " + description + " " + protocol).lower()

        # Preliminary classification based on assay methodology
        if any(term in text for term in ["sirna", "rna interference", "knockdown"]):
            classification = "Genetic screening"

        elif any(term in text for term in [
            "binding affinity", "enzyme inhibition",
            "biochemical assay", "inhibitor"
        ]):
            classification = "Potential compound assay - verify"

        elif any(term in text for term in [
            "cell viability", "cytotoxicity", "proliferation"
        ]):
            classification = "Cell-based assay - verify target relevance"

        else:
            classification = "Unclassified - inspect"

        pubchem_classification.append({
            "AID": aid,
            "Assay_Name": name,
            "Classification": classification,
            "Description": description[:500]
        })

pubchem_df = pd.DataFrame(pubchem_classification)

print("Total assay records:", len(pubchem_df))

print("\nClassification summary:")
print(pubchem_df["Classification"].value_counts())

print("\nComplete assay classification:")
display(
    pubchem_df[
        ["AID", "Assay_Name", "Classification"]
    ].sort_values("AID")
)

pubchem_df.to_csv(
    "PAPSS2_PubChem_assay_classification.csv",
    index=False
)

print("\nSaved: PAPSS2_PubChem_assay_classification.csv")


Total assay records: 49

Classification summary:
Classification
Genetic screening                    47
Unclassified - inspect                1
Potential compound assay - verify     1
Name: count, dtype: int64

Complete assay classification:


,AID,Assay_Name,Classification
0,1622,RNAi Global Initiative pilot viability screen ...,Genetic screening
1,1904,siRNA Circadian Assay,Genetic screening
2,624099,An siRNA screen for human genes that are invol...,Genetic screening
3,651810,Genome-wide high-content siRNA screens for reg...,Genetic screening
4,651811,Genome-wide high-content siRNA screens for reg...,Genetic screening
5,720703,AT1R-EGFR transactivation kinome screen - primary,Genetic screening
6,743121,Druggable Genome siRNA Screen for Enhancers of...,Genetic screening
7,743454,RNAi screen for vorinostat resistance genes - ...,Unclassified - inspect
8,1053208,A screen to identify the genes that modify the...,Genetic screening
9,1117269,Human kinome siRNA screen in A549-BTR and MDA-...,Genetic screening



Saved: PAPSS2_PubChem_assay_classification.csv


In [ ]:

import requests

for aid in [1454491, 743454]:

    print("\n" + "="*75)
    print("PUBCHEM ASSAY:", aid)

    # Retrieve original assay description
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{aid}/description/JSON"
    r = requests.get(url, timeout=60)

    print("\nDESCRIPTION STATUS:", r.status_code)

    if r.status_code == 200:
        data = r.json()
        print(str(data)[:5000])
    else:
        print(r.text[:1000])

    # Retrieve assay data
    data_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{aid}/CSV"
    d = requests.get(data_url, timeout=90)

    print("\nACTIVITY DATA STATUS:", d.status_code)

    if d.status_code == 200:
        lines = d.text.splitlines()

        print("Number of CSV lines:", len(lines))
        print("\nFirst 15 lines:")
        print("\n".join(lines[:15]))

        with open(f"PubChem_AID_{aid}.csv", "w", encoding="utf-8") as f:
            f.write(d.text)

        print(f"\nSaved PubChem_AID_{aid}.csv")

    else:
        print(d.text[:1000])



PUBCHEM ASSAY: 1454491

DESCRIPTION STATUS: 200
{'PC_AssayContainer': [{'assay': {'descr': {'aid': {'id': 1454491, 'version': 1}, 'aid_source': {'db': {'name': 'ChEMBL', 'source_id': {'str': 'CHEMBL4036829'}}}, 'name': 'Binding affinity to SK2 (unknown origin)', 'description': ['Title: Discovery of a Series of Indazole TRPA1 Antagonists.', '', 'Abstract: A series of TRPA1 antagonists is described which has as its core structure an indazole moiety. The physical properties and in vitro DMPK profiles are discussed. Good in vivo exposure was obtained with several analogs, allowing efficacy to be assessed in rodent models of inflammatory pain. Two compounds showed significant activity in these models when administered either systemically or topically. Protein chimeras were constructed to indicate compounds from the series bound in the S5 region of the channel, and a computational docking model was used to propose a binding mode for example compounds.'], 'comment': ['Journal: ACS Med Chem L